# Prévision de séries temporelles : modèles classiques vs modèles de fondation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dbetteb/early-ML/blob/master/13_SERIES_TEMPORELLES_FONDATION/Prevision_modeles_fondation_darts_TabICL.ipynb)

## Objectif

Prévoir la **température de l'air heure par heure, 24 h à l'avance**, à partir de son historique, et comparer :

| famille | modèle | apprentissage sur nos données ? |
|---|---|---|
| référence | persistance J−1 (« demain = aujourd'hui ») | non |
| statistique | Holt-Winters (lissage exponentiel saisonnier) | oui, ré-estimé à chaque prévision |
| apprentissage automatique | LightGBM sur valeurs passées + heure/jour | oui, une fois |
| **fondation** (séries temporelles) | **Chronos-2** (Amazon, 2025) via `darts` | **non** (zéro-shot) |
| **fondation** (tabulaire) | **TabICL** (Inria, 2025) en mode prévision | **non** (apprentissage en contexte) |

Un **modèle de fondation** est un grand réseau pré-entraîné sur un énorme corpus de séries (réelles et synthétiques).
On lui donne l'historique récent (le *contexte*), et il produit directement des prévisions probabilistes (des quantiles), sans aucun
entraînement sur nos données. Tout tourne sur **CPU** en une dizaine de minutes.

**Données** : station météo du Max Planck Institute for Biogeochemistry (Iéna), année 2020, pas de 10 min,
fournies par `darts` (`WeatherDataset`).

## 1. Installation et imports

Sur Colab, `torch` est déjà installé : il suffit d'ajouter `darts` (avec ses dépendances torch) et `tabicl` (avec l'extra `forecast`).
Les poids des modèles de fondation (Chronos-2 : 120 M de paramètres, environ 480 Mo ; TabICL : environ 100 Mo) sont téléchargés depuis
Hugging Face au premier appel.

In [ ]:
%pip install -q "darts[torch]" "tabicl[forecast]"

In [ ]:
import os, time, warnings, logging
os.environ["TQDM_DISABLE"] = "1"          # supprime les barres de progression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from darts import TimeSeries
from darts.datasets import WeatherDataset
from darts.models import ExponentialSmoothing, LightGBMModel, Chronos2Model
from darts.utils.likelihood_models import QuantileRegression
from darts.utils.utils import ModelMode, SeasonalityMode

warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.ERROR)
torch.set_num_threads(max(1, torch.get_num_threads()))
print("torch", torch.__version__, "| GPU :", torch.cuda.is_available())

## 2. Données

On moyenne les mesures à 10 min sur chaque heure : 8 784 valeurs horaires pour l'année 2020 (bissextile).

In [ ]:
meteo = WeatherDataset().load()
temp = meteo["T (degC)"].to_series().resample("h").mean()
temp = temp.loc[:"2020-12-31 23:00"].interpolate()        # dernière heure incomplète + éventuels trous
temp.name = "T"
serie = TimeSeries.from_series(temp.astype(np.float32))
print(serie.n_timesteps, "heures, de", serie.start_time(), "à", serie.end_time())

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5), gridspec_kw={"width_ratios": [3, 1]})
temp.plot(ax=ax[0], lw=0.5); ax[0].set_ylabel("T (°C)"); ax[0].set_title("Température horaire, Iéna 2020")
temp.loc["2020-07-01":"2020-07-07"].plot(ax=ax[1]); ax[1].set_title("Zoom sur une semaine")
plt.tight_layout(); plt.show()

La série combine une **saison annuelle**, un **cycle diurne** (période 24 h) dont l'amplitude varie, et des épisodes météorologiques
de quelques jours, non périodiques. C'est cette dernière composante qui rend la prévision à 24 h difficile.

## 3. Protocole : prévisions glissantes

On simule une utilisation opérationnelle sur les **28 derniers jours** de l'année : chaque jour à minuit, on prévoit les 24 heures suivantes
avec les seules données disponibles avant minuit (*rolling origin*). Cela donne 28 × 24 = 672 couples (prévision, observation).

Indicateurs :
* **MAE** et **RMSE** de la prévision centrale (médiane) ;
* **MASE** : MAE divisée par celle de la persistance J−1 sur la période d'apprentissage (MASE < 1 : meilleur que la persistance) ;
* **perte quantile moyenne** sur les quantiles 10 %, 50 % et 90 % (approximation grossière du CRPS) : juge la prévision probabiliste ;
* **couverture** de l'intervalle [q10, q90], qui devrait être proche de 80 %.

In [ ]:
H = 24                                     # horizon (heures)
N_JOURS_TEST = 28
QUANTILES = [0.1, 0.5, 0.9]

debut_test = serie.end_time() - pd.Timedelta(hours=N_JOURS_TEST * H - 1)
origines = pd.date_range(debut_test, periods=N_JOURS_TEST, freq="D")
historique = serie.drop_after(debut_test)  # tout ce qui précède la première origine
print("première origine :", origines[0], "| dernière :", origines[-1])

y_obs = np.stack([temp.loc[o: o + pd.Timedelta(hours=H - 1)].to_numpy() for o in origines])   # (28, 24)
h_np = historique.values().ravel()
echelle_mase = np.mean(np.abs(h_np[24:] - h_np[:-24]))        # MAE de la persistance J-1 sur l'historique

def pinball(y, q, tau):
    return np.mean(np.maximum(tau * (y - q), (tau - 1) * (y - q)))

def scores(Q):
    """Q : tableau (n_origines, H, 3) des quantiles 10/50/90 %."""
    med = Q[..., 1]
    return {"MAE": np.mean(np.abs(med - y_obs)),
            "RMSE": np.sqrt(np.mean((med - y_obs) ** 2)),
            "MASE": np.mean(np.abs(med - y_obs)) / echelle_mase,
            "perte quantile": np.mean([pinball(y_obs, Q[..., k], t) for k, t in enumerate(QUANTILES)]),
            "couverture 80 %": np.mean((y_obs >= Q[..., 0]) & (y_obs <= Q[..., 2]))}

PREVISIONS, TEMPS = {}, {}                 # nom du modèle -> tableau (28, 24, 3)

def depuis_darts(liste_ts):
    """Convertit la liste de prévisions darts (composantes T_q0.100, ...) en tableau (n, H, 3)."""
    return np.stack([ts.values()[:H] for ts in liste_ts])

## 4. Modèles de référence

### 4.1 Persistance J−1 probabiliste

Prévision : la valeur observée 24 h plus tôt. Pour obtenir des quantiles, on ajoute les quantiles empiriques des variations
sur 24 h calculées sur l'historique. C'est la référence minimale à battre.

In [ ]:
t0 = time.perf_counter()
dq = np.quantile(h_np[24:] - h_np[:-24], QUANTILES)
veille = np.stack([temp.loc[o - pd.Timedelta(hours=H): o - pd.Timedelta(hours=1)].to_numpy() for o in origines])
PREVISIONS["Persistance J-1"] = veille[..., None] + dq
TEMPS["Persistance J-1"] = time.perf_counter() - t0
scores(PREVISIONS["Persistance J-1"])

### 4.2 Holt-Winters

Lissage exponentiel avec tendance et saisonnalité additives de période 24 h, **ré-estimé** à chaque origine sur les 60 derniers jours.
Les quantiles sont obtenus par simulation (200 trajectoires).

In [ ]:
t0 = time.perf_counter()
hw = ExponentialSmoothing(trend=ModelMode.ADDITIVE, damped=True, seasonal=SeasonalityMode.ADDITIVE, seasonal_periods=24)
prev_hw = hw.historical_forecasts(serie, start=origines[0], forecast_horizon=H, stride=H, retrain=True,
                                  train_length=60 * 24, num_samples=200, last_points_only=False, verbose=False)
PREVISIONS["Holt-Winters"] = np.stack([np.quantile(p.all_values()[:H, 0, :], QUANTILES, axis=1).T for p in prev_hw])
TEMPS["Holt-Winters"] = time.perf_counter() - t0
scores(PREVISIONS["Holt-Winters"])

### 4.3 LightGBM

Modèle de *gradient boosting* qui prédit directement les 24 heures suivantes à partir des 48 dernières valeurs et d'un encodage
cyclique de l'heure et du jour de l'année. Il est entraîné **une seule fois** sur l'historique (régression quantile : un modèle par quantile).

In [ ]:
t0 = time.perf_counter()
lgbm = LightGBMModel(lags=48, lags_future_covariates=[0], output_chunk_length=H,
                     add_encoders={"cyclic": {"future": ["hour", "dayofyear"]}},
                     likelihood="quantile", quantiles=QUANTILES, n_estimators=300, learning_rate=0.05,
                     num_leaves=31, random_state=0, verbose=-1)
lgbm.fit(historique)
prev_lgbm = lgbm.historical_forecasts(serie, start=origines[0], forecast_horizon=H, stride=H, retrain=False,
                                      predict_likelihood_parameters=True, last_points_only=False, verbose=False)
PREVISIONS["LightGBM"] = depuis_darts(prev_lgbm)
TEMPS["LightGBM"] = time.perf_counter() - t0
scores(PREVISIONS["LightGBM"])

## 5. Modèles de fondation

### 5.1 Chronos-2 dans `darts` (zéro-shot)

[Chronos-2](https://huggingface.co/amazon/chronos-2) (Ansari *et al.*, 2025) est un encodeur de type transformeur qui découpe la série
en *patchs* de 16 pas et prédit directement 21 quantiles. Il accepte jusqu'à 8 192 pas de contexte, des séries multivariées et des covariables.

Dans `darts`, il s'utilise comme n'importe quel autre modèle. `fit()` est obligatoire pour respecter l'interface, mais il **n'entraîne rien** :
il charge seulement les poids pré-entraînés. Remplacer `Chronos2Model` par `TimesFM2p5Model`, `TimesFM3Model`, `TiRex2Model` ou `PatchTSTFMModel` suffit pour essayer
d'autres modèles de fondation (certains demandent un paquet supplémentaire, voir la documentation `darts`).

In [ ]:
CHRONOS_LOCAL = None          # dossier contenant config.json + model.safetensors si les poids sont déjà téléchargés

def chronos(contexte):
    return Chronos2Model(input_chunk_length=contexte, output_chunk_length=H, local_dir=CHRONOS_LOCAL,
                         likelihood=QuantileRegression(QUANTILES), random_state=0,
                         pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": False})

t0 = time.perf_counter()
chronos2 = chronos(2048).fit(historique)      # aucun apprentissage : chargement des poids
prev_c2 = chronos2.historical_forecasts(serie, start=origines[0], forecast_horizon=H, stride=H, retrain=False,
                                        predict_likelihood_parameters=True, last_points_only=False, verbose=False)
PREVISIONS["Chronos-2 (zéro-shot)"] = depuis_darts(prev_c2)
TEMPS["Chronos-2 (zéro-shot)"] = time.perf_counter() - t0
scores(PREVISIONS["Chronos-2 (zéro-shot)"])

### 5.2 TabICL : un modèle de fondation *tabulaire* utilisé pour la prévision

Approche très différente : on transforme la série en un **tableau** dont chaque ligne est un instant, avec des variables explicatives
calculées à partir de la date (indice temporel, heure, jour, composantes de Fourier des périodes détectées automatiquement), et la valeur
à prédire. Les lignes passées forment le contexte, les lignes futures sont à compléter. Le transformeur tabulaire pré-entraîné
[TabICL](https://github.com/soda-inria/tabicl) prédit alors les cases manquantes **en contexte**, comme dans le benchmark SER.

On limite le contexte à 1 024 heures (environ 6 semaines) pour garder un temps de calcul raisonnable sur CPU
(quelques secondes par prévision, donc quelques minutes pour les 28 origines).

In [ ]:
from tabicl.forecast import TabICLForecaster

TABICL_CHECKPOINT = None      # chemin local facultatif vers le checkpoint
tabicl_fc = TabICLForecaster(max_context_length=1024,
                             tabicl_config={"device": "cpu", "model_path": TABICL_CHECKPOINT})

t0 = time.perf_counter()
sorties = []
for o in origines:
    contexte = temp.loc[: o - pd.Timedelta(hours=1)].reset_index()
    contexte.columns = ["timestamp", "target"]
    p = tabicl_fc.predict_df(contexte, prediction_length=H, quantiles=QUANTILES)
    sorties.append(p[QUANTILES].to_numpy())
PREVISIONS["TabICL (zéro-shot)"] = np.stack(sorties)
TEMPS["TabICL (zéro-shot)"] = time.perf_counter() - t0
scores(PREVISIONS["TabICL (zéro-shot)"])

## 6. Comparaison

In [ ]:
tab = pd.DataFrame({m: scores(Q) for m, Q in PREVISIONS.items()}).T
tab["temps total (s)"] = pd.Series(TEMPS)
tab.sort_values("perte quantile").style.format("{:.3f}").background_gradient(subset=["MASE"], cmap="RdYlGn_r")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for m, Q in PREVISIONS.items():
    ax[0].plot(np.arange(1, H + 1), np.mean(np.abs(Q[..., 1] - y_obs), axis=0), marker="o", ms=3, label=m)
ax[0].set_xlabel("échéance (h)"); ax[0].set_ylabel("MAE (°C)"); ax[0].set_title("Erreur selon l'échéance")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

jours = slice(10, 17)                       # une semaine de la période de test
t_axe = pd.date_range(origines[jours.start], periods=(jours.stop - jours.start) * H, freq="h")
ax[1].plot(t_axe, y_obs[jours].ravel(), "k", lw=2, label="observé")
for m in ["Persistance J-1", "Chronos-2 (zéro-shot)", "TabICL (zéro-shot)"]:
    Q = PREVISIONS[m][jours].reshape(-1, 3)
    l, = ax[1].plot(t_axe, Q[:, 1], lw=1.2, label=m)
    ax[1].fill_between(t_axe, Q[:, 0], Q[:, 2], color=l.get_color(), alpha=0.15)
for o in origines[jours]: ax[1].axvline(o, color="gray", lw=0.3)
ax[1].set_title("Prévisions à 24 h et intervalles 80 %"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. L'importance du contexte

Un modèle de fondation n'apprend rien sur nos données : **tout ce qu'il sait de la série vient du contexte** qu'on lui fournit.
On fait varier la longueur de contexte de Chronos-2, d'un jour à près de trois mois.

In [ ]:
res_ctx = {}
for L in [24, 72, 168, 672, 2048]:
    t0 = time.perf_counter()
    m = chronos(L).fit(historique)
    p = m.historical_forecasts(serie, start=origines[0], forecast_horizon=H, stride=H, retrain=False,
                               predict_likelihood_parameters=True, last_points_only=False, verbose=False)
    res_ctx[L] = scores(depuis_darts(p))
    print(f"contexte {L:5d} h : MAE = {res_ctx[L]['MAE']:.3f} °C, perte quantile = {res_ctx[L]['perte quantile']:.3f}"
          f"  ({time.perf_counter() - t0:.0f} s)")

ref = scores(PREVISIONS["Persistance J-1"])
pd.DataFrame(res_ctx).T[["MAE", "perte quantile"]].plot(marker="o", logx=True, subplots=True, layout=(1, 2), figsize=(12, 3.5),
                                                       title=["MAE", "perte quantile"], legend=False)
plt.gcf().axes[0].axhline(ref["MAE"], ls="--", c="gray", label="persistance J-1")
plt.gcf().axes[1].axhline(ref["perte quantile"], ls="--", c="gray", label="persistance J-1")
plt.gcf().axes[0].legend()
for a in plt.gcf().axes: a.set_xlabel("longueur de contexte (h)")
plt.tight_layout(); plt.show()

Avec un contexte d'un seul jour, le modèle ne peut pas distinguer le cycle diurne des fluctuations météorologiques :
on s'attend à une nette dégradation quand le contexte raccourcit. 

## 8. Points de vigilance

* **Contamination** : ce jeu de données (« Weather ») fait partie des benchmarks de prévision les plus utilisés. Il a pu être vu
  pendant le pré-entraînement, de même que des données très proches (réanalyses ERA5). Pour une évaluation sérieuse, il faut des
  données postérieures à la date de pré-entraînement ou non publiques.
* **Une seule série, 28 origines** : les écarts entre modèles sont bruités. Il faudrait des intervalles de confiance (bootstrap par
  blocs de jours) et un test apparié avant de conclure.
* **Calibration** : la couverture des intervalles 80 % peut s'écarter de 80 %. Une recalibration conforme causale (décalage des
  quantiles à partir des erreurs passées) corrige cela à peu de frais, sans toucher au modèle.

## 9. Exercices

1. **Covariables** : Chronos-2 accepte des covariables passées (`past_covariates`) et futures connues (`future_covariates`).
   Ajouter le rayonnement solaire mesuré `SWDR (W/m²)` comme covariable passée. Le gain est-il visible à 24 h ?
2. **Autre modèle de fondation** : remplacer `Chronos2Model` par `TimesFM2p5Model` (Google) ou `TiRex2Model` (NX-AI). Comparer précision et temps de calcul.
3. **Horizon** : passer à `H = 72`. Comment évolue l'écart entre les modèles de fondation et la persistance ?
4. **Ajustement fin** (sur GPU) : `darts` permet le *fine-tuning*, complet ou par LoRA, des modèles de fondation (`enable_finetuning=True`).
   Ajuster Chronos-2 sur les mois de janvier à octobre et comparer au zéro-shot.

## Références

* A. F. Ansari *et al.* (2025), *Chronos-2: From Univariate to Universal Forecasting*, arXiv:2510.15821.
* Z. Dai, D. Bader, A. Gysi (2026), *Unified Zero-Shot Time Series Forecasting: A Darts Foundation*, arXiv:2606.27438.
* J. Qu, D. Holzmüller, G. Varoquaux, M. Le Morvan (2025), *TabICL: A tabular foundation model for in-context learning on large data*, ICML.
* S. B. Hoo *et al.* (2025), *From tables to time: how TabPFN-v2 outperforms specialized time series forecasting models*, arXiv:2501.02945.
* R. J. Hyndman, G. Athanasopoulos (2021), *Forecasting: Principles and Practice*, 3e éd., OTexts (MASE, prévisions glissantes).
* Documentation `darts` : [modèles de fondation](https://unit8co.github.io/darts/examples/25-FoundationModel-examples.html).